In [1]:
# ============================================================
# 1. Configuração do projeto — NDMI
# ============================================================

from pathlib import Path

# Diretório principal do projeto
projeto = Path(
    r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao"
)

# Pasta de saída dos produtos NDMI
pasta_ndmi = (
    projeto
    / "data"
    / "processed"
    / "indices"
    / "ndmi"
)

# Cria a pasta caso ainda não exista
pasta_ndmi.mkdir(
    parents=True,
    exist_ok=True
)

print("✓ Projeto:", projeto)
print("✓ Pasta de saída NDMI:", pasta_ndmi)

✓ Projeto: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao
✓ Pasta de saída NDMI: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\indices\ndmi


In [2]:
# ============================================================
# 2. Bibliotecas e conexão com Google Earth Engine
# ============================================================

import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd

# Inicializa o Google Earth Engine
try:
    ee.Initialize()
    print("✓ Google Earth Engine conectado")

except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("✓ Google Earth Engine autenticado e conectado")

✓ Google Earth Engine conectado


In [3]:
# ============================================================
# 3. Área de Interesse (AOI)
# ============================================================

# Pasta onde estão os dados brutos
pasta_raw = projeto / "data" / "raw"

# Procura automaticamente o shapefile da Área do Imóvel
arquivos_imovel = list(pasta_raw.rglob("Area_do_Imovel.shp"))

print("Shapefiles encontrados:")

for arquivo in arquivos_imovel:
    print(" ", arquivo)

if not arquivos_imovel:
    raise FileNotFoundError(
        "❌ O shapefile Area_do_Imovel.shp não foi encontrado."
    )

# Usa o primeiro shapefile encontrado
arquivo_imovel = arquivos_imovel[0]

print()
print("✓ Arquivo utilizado:")
print(arquivo_imovel)

# Leitura da camada
imovel = gpd.read_file(arquivo_imovel)

# Usa o primeiro registro, pois os dois registros
# possuem a mesma geometria
aoi = imovel.geometry.iloc[0]

print()
print("✓ AOI criada")
print("Tipo de geometria:", aoi.geom_type)
print("CRS original:", imovel.crs)

# Conversão para Google Earth Engine
aoi_geojson = aoi.__geo_interface__
aoi_ee = ee.Geometry(aoi_geojson)

print("✓ AOI convertida para Google Earth Engine")
print("Tipo:", aoi_ee.type().getInfo())

Shapefiles encontrados:
  C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\raw\Area_do_Imovel\Area_do_Imovel.shp

✓ Arquivo utilizado:
C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\raw\Area_do_Imovel\Area_do_Imovel.shp

✓ AOI criada
Tipo de geometria: Polygon
CRS original: EPSG:4674
✓ AOI convertida para Google Earth Engine
Tipo: Polygon


In [4]:
# ============================================================
# 4. Coleção Sentinel-2 — NDMI
# ============================================================

ano = 2025

colecao_ano = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_ee)
    .filterDate(
        f"{ano}-01-01",
        f"{ano + 1}-01-01"
    )
    .filter(
        ee.Filter.lt(
            "CLOUDY_PIXEL_PERCENTAGE",
            20
        )
    )
)

quantidade = colecao_ano.size().getInfo()

print("✓ Coleção Sentinel-2 criada")
print("Ano:", ano)
print("Imagens encontradas:", quantidade)

✓ Coleção Sentinel-2 criada
Ano: 2025
Imagens encontradas: 99


In [5]:
# ============================================================
# 5. Seleção mensal — cobertura completa da AOI
# ============================================================

imagens_mensais = []

# Área total da AOI
area_aoi = aoi_ee.area(1)


# ------------------------------------------------------------
# Função para calcular a cobertura da imagem sobre a AOI
# ------------------------------------------------------------

def calcular_cobertura_aoi(imagem):

    intersecao = imagem.geometry().intersection(
        aoi_ee,
        1
    )

    area_intersecao = intersecao.area(1)

    cobertura = (
        area_intersecao
        .divide(area_aoi)
        .multiply(100)
    )

    return imagem.set(
        "COBERTURA_AOI",
        cobertura
    )


# ------------------------------------------------------------
# Seleção da melhor imagem de cada mês
# ------------------------------------------------------------

for mes in range(1, 13):

    inicio = f"{ano}-{mes:02d}-01"

    if mes == 12:
        fim = f"{ano + 1}-01-01"
    else:
        fim = f"{ano}-{mes + 1:02d}-01"


    # Imagens do mês + cálculo da cobertura
    colecao_mes = (
        colecao_ano
        .filterDate(inicio, fim)
        .map(calcular_cobertura_aoi)
    )


    # Mantém somente imagens que cobrem
    # pelo menos 99,99% da propriedade
    colecao_completa = (
        colecao_mes
        .filter(
            ee.Filter.gte(
                "COBERTURA_AOI",
                99.99
            )
        )
        .sort(
            "CLOUDY_PIXEL_PERCENTAGE"
        )
    )


    quantidade = colecao_completa.size().getInfo()


    # --------------------------------------------------------
    # Nenhuma imagem adequada
    # --------------------------------------------------------

    if quantidade == 0:

        quantidade_total = (
            colecao_mes.size().getInfo()
        )

        if quantidade_total == 0:

            print(
                f"⚠️ {mes:02d}/{ano} — "
                "nenhuma imagem encontrada"
            )

        else:

            print(
                f"⚠️ {mes:02d}/{ano} — "
                "nenhuma imagem cobre 99,99% da AOI"
            )

        continue


    # --------------------------------------------------------
    # Melhor imagem do mês
    # --------------------------------------------------------

    imagem_mes = ee.Image(
        colecao_completa.first()
    )


    data = (
        ee.Date(
            imagem_mes.get(
                "system:time_start"
            )
        )
        .format("YYYY-MM-dd")
        .getInfo()
    )


    nuvens = imagem_mes.get(
        "CLOUDY_PIXEL_PERCENTAGE"
    ).getInfo()


    cobertura = imagem_mes.get(
        "COBERTURA_AOI"
    ).getInfo()


    tile = imagem_mes.get(
        "MGRS_TILE"
    ).getInfo()


    imagem_id = imagem_mes.get(
        "PRODUCT_ID"
    ).getInfo()


    # Guarda informações da imagem
    imagens_mensais.append({

        "mes": mes,

        "data": data,

        "nuvens": nuvens,

        "cobertura_aoi": cobertura,

        "tile": tile,

        "id": imagem_id,

        "imagem": imagem_mes

    })


    print(
        f"✓ {mes:02d}/{ano} | "
        f"{data} | "
        f"nuvens: {nuvens:.2f}% | "
        f"AOI: {cobertura:.2f}% | "
        f"tile: {tile}"
    )


print()
print(
    f"✓ Total de imagens selecionadas: "
    f"{len(imagens_mensais)}"
)

✓ 01/2025 | 2025-01-08 | nuvens: 3.87% | AOI: 100.00% | tile: 22KHV
✓ 02/2025 | 2025-02-17 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 03/2025 | 2025-03-09 | nuvens: 0.01% | AOI: 100.00% | tile: 22KHV
✓ 04/2025 | 2025-04-08 | nuvens: 0.02% | AOI: 100.00% | tile: 22KHV
✓ 05/2025 | 2025-05-13 | nuvens: 8.10% | AOI: 100.00% | tile: 23KKQ
✓ 06/2025 | 2025-06-17 | nuvens: 0.00% | AOI: 100.00% | tile: 23KKQ
✓ 07/2025 | 2025-07-27 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 08/2025 | 2025-08-01 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 09/2025 | 2025-09-15 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 10/2025 | 2025-10-05 | nuvens: 0.00% | AOI: 100.00% | tile: 22KHV
✓ 11/2025 | 2025-11-21 | nuvens: 0.04% | AOI: 100.00% | tile: 22KHV
✓ 12/2025 | 2025-12-04 | nuvens: 1.61% | AOI: 100.00% | tile: 22KHV

✓ Total de imagens selecionadas: 12


In [6]:
# ============================================================
# 6. Cálculo do NDMI mensal
# ============================================================

for item in imagens_mensais:

    imagem = item["imagem"]

    # NDMI = (B8 - B11) / (B8 + B11)
    ndmi = (
        imagem
        .normalizedDifference(["B8", "B11"])
        .rename("NDMI")
        .clip(aoi_ee)
    )

    # Guarda o NDMI junto com as informações da imagem
    item["ndmi"] = ndmi

    print(
        f"✓ {item['mes']:02d}/{ano} | "
        f"{item['data']} | "
        f"NDMI calculado"
    )

print()
print(
    f"✓ NDMI calculado para "
    f"{len(imagens_mensais)} meses"
)

✓ 01/2025 | 2025-01-08 | NDMI calculado
✓ 02/2025 | 2025-02-17 | NDMI calculado
✓ 03/2025 | 2025-03-09 | NDMI calculado
✓ 04/2025 | 2025-04-08 | NDMI calculado
✓ 05/2025 | 2025-05-13 | NDMI calculado
✓ 06/2025 | 2025-06-17 | NDMI calculado
✓ 07/2025 | 2025-07-27 | NDMI calculado
✓ 08/2025 | 2025-08-01 | NDMI calculado
✓ 09/2025 | 2025-09-15 | NDMI calculado
✓ 10/2025 | 2025-10-05 | NDMI calculado
✓ 11/2025 | 2025-11-21 | NDMI calculado
✓ 12/2025 | 2025-12-04 | NDMI calculado

✓ NDMI calculado para 12 meses


In [7]:
# ============================================================
# 7. Estatísticas do NDMI mensal
# ============================================================

print("=" * 70)
print(f"ESTATÍSTICAS NDMI — {ano}")
print("=" * 70)

for item in imagens_mensais:

    ndmi = item["ndmi"]

    estatisticas = ndmi.reduceRegion(
        reducer=(
            ee.Reducer.mean()
            .combine(
                reducer2=ee.Reducer.median(),
                sharedInputs=True
            )
            .combine(
                reducer2=ee.Reducer.minMax(),
                sharedInputs=True
            )
        ),
        geometry=aoi_ee,
        scale=20,
        maxPixels=1e9
    ).getInfo()

    print(
        f"{item['data']} | "
        f"média: {estatisticas.get('NDMI_mean', float('nan')):.4f} | "
        f"mediana: {estatisticas.get('NDMI_median', float('nan')):.4f} | "
        f"mín: {estatisticas.get('NDMI_min', float('nan')):.4f} | "
        f"máx: {estatisticas.get('NDMI_max', float('nan')):.4f}"
    )

ESTATÍSTICAS NDMI — 2025
2025-01-08 | média: 0.0562 | mediana: 0.0097 | mín: -0.2647 | máx: 0.5181
2025-02-17 | média: 0.3384 | mediana: 0.3730 | mín: -0.1409 | máx: 0.4790
2025-03-09 | média: 0.1953 | mediana: 0.1816 | mín: -0.1448 | máx: 0.4897
2025-04-08 | média: -0.0131 | mediana: -0.1189 | mín: -0.3276 | máx: 0.5014
2025-05-13 | média: -0.0367 | mediana: -0.0839 | mín: -0.3601 | máx: 0.5063
2025-06-17 | média: 0.0934 | mediana: 0.1425 | mín: -0.3351 | máx: 0.5448
2025-07-27 | média: 0.1219 | mediana: 0.1344 | mín: -0.3249 | máx: 0.5199
2025-08-01 | média: 0.1301 | mediana: 0.1347 | mín: -0.2945 | máx: 0.4966
2025-09-15 | média: -0.0779 | mediana: -0.1387 | mín: -0.3021 | máx: 0.4122
2025-10-05 | média: -0.1142 | mediana: -0.2128 | mín: -0.3079 | máx: 0.4036
2025-11-21 | média: -0.0470 | mediana: -0.0879 | mín: -0.3089 | máx: 0.4134
2025-12-04 | média: -0.0502 | mediana: -0.0730 | mín: -0.3021 | máx: 0.4548


In [8]:
# ============================================================
# 8. Exportação dos NDMI mensais — GeoTIFF
# ============================================================

for item in imagens_mensais:

    ndmi = item["ndmi"]

    data = pd.to_datetime(item["data"])
    data_str = data.strftime("%Y%m%d")

    arquivo_saida = (
        pasta_ndmi / f"ndmi_{data_str}.tif"
    )

    print(
        f"Exportando {data.strftime('%d/%m/%Y')}..."
    )

    geemap.ee_export_image(
        ndmi,
        filename=str(arquivo_saida),
        scale=20,
        region=aoi_ee,
        file_per_band=False
    )

    print(
        f"✓ Salvo: {arquivo_saida.name}"
    )

print()
print("✓ Exportação dos NDMI concluída.")

Exportando 08/01/2025...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\indices\ndmi\ndmi_20250108.tif
✓ Salvo: ndmi_20250108.tif
Exportando 17/02/2025...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\indices\ndmi\ndmi_20250217.tif
✓ Salvo: ndmi_20250217.tif
Exportando 09/03/2025...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\indices\ndmi\ndmi_20250309.tif
✓ Salvo: ndmi_20250309.tif
Exportando 08/04/2025...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\indices\ndmi\ndmi_20250408.tif
✓ Salvo: ndmi_20250408.tif
Exportando 13/05/2025...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetaca